# Urban Intelligence Platform: Traffic Congestion Prediction & K-Means Clustering
**Chennai Smart City Operational Mobility Intelligence**

This notebook trains:
1. A **RandomForestClassifier** with a scikit-learn Pipeline on `Traffic.csv` to classify `Traffic Situation` (`low`, `normal`, `high`, `heavy`).
2. A **K-Means Clustering Pipeline** (`StandardScaler` + `KMeans(n_clusters=4)`) to discover traffic density archetypes based on multimodal vehicle counts.
3. Exports `traffic_random_forest_pipeline.joblib` and `traffic_kmeans_pipeline.joblib` for real-time FastAPI inference.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


## 1. Load and Inspect Traffic Dataset

In [ ]:
df = pd.read_csv('Traffic.csv')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print('\nMissing values:\n', df.isnull().sum())
print('\nTraffic Situation distribution:\n', df['Traffic Situation'].value_counts())
df.head()

## 2. Feature Engineering & Preprocessing Pipeline

In [ ]:
def parse_time_to_hour(val):
    try:
        return pd.to_datetime(str(val).strip()).hour
    except Exception:
        return 12

df['Hour'] = df['Time'].apply(parse_time_to_hour)
df['Traffic Situation'] = df['Traffic Situation'].str.strip().str.lower()

feature_cols = ['Date', 'Day of the week', 'Hour', 'CarCount', 'BikeCount', 'BusCount', 'TruckCount', 'Total']
X = df[feature_cols].copy()
y = df['Traffic Situation'].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)


## 3. Train Random Forest Classifier Pipeline

In [ ]:
cat_cols = ['Day of the week']
num_cols = ['Date', 'Hour', 'CarCount', 'BikeCount', 'BusCount', 'TruckCount', 'Total']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', 'passthrough', num_cols)
])

rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, max_depth=16, random_state=42, n_jobs=-1))
])

rf_pipeline.fit(X_train, y_train)
y_pred = rf_pipeline.predict(X_test)

print('Test Accuracy:', round(accuracy_score(y_test, y_pred), 4))
print('\nClassification Report:\n', classification_report(y_test, y_pred))
print('\nConfusion Matrix:\n', confusion_matrix(y_test, y_pred))


## 4. Train K-Means Clustering on Vehicle Attributes

In [ ]:
cluster_features = ['CarCount', 'BikeCount', 'BusCount', 'TruckCount', 'Total']
kmeans_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans', KMeans(n_clusters=4, random_state=42, n_init=10))
])

df['Cluster'] = kmeans_pipeline.fit_predict(df[cluster_features])
print('Cluster counts:\n', df['Cluster'].value_counts())
print('\nCluster feature means:\n', df.groupby('Cluster')[cluster_features].mean())


## 5. Export Pipelines for FastAPI Serving

In [ ]:
joblib.dump({'pipeline': rf_pipeline}, 'traffic_random_forest_pipeline.joblib')
joblib.dump({'pipeline': kmeans_pipeline}, 'traffic_kmeans_pipeline.joblib')
print('Traffic pipelines exported successfully!')
